# 10 Transformer Ablation Pilot

Small ablations for selected decoding parameters:
- horizon in {3,4,5}
- top-k in {4,8}
- beam width in {2,4}
- deterministic vs stochastic branch choice

This notebook runs lightweight diagnostic comparisons on one teacher and one prompt bank.

In [ ]:
from pathlib import Path
import json
import itertools
import sys

import pandas as pd
import torch

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parents[2] if NB_DIR.name == "active" else Path.cwd().resolve()
SRC = ROOT / "GitHub" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from drift_selection.selected_decoding import SelectedDecodingConfig, selected_next_token
from drift_selection.transformer import load_model_config
from drift_selection.training import load_trained_model
from drift_selection.transformer_pipeline import load_prompt_bank, ensure_tokenizer_and_encoded_splits, load_main_config

cfg = load_main_config(ROOT, "GitHub/configs/transformer_teacher_student.yaml")
_ = ensure_tokenizer_and_encoded_splits(ROOT, cfg, force=False)
out_root = ROOT / cfg["paths"]["root_outputs_dir"]

teacher_summary = json.loads((out_root / "teacher/training_summary.json").read_text())
teacher_ckpt = teacher_summary["history"]["best_checkpoint_path"] or teacher_summary["history"]["final_checkpoint_path"]
model_cfg = load_model_config(out_root / "teacher/model_config.json")
device = torch.device("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
teacher = load_trained_model(Path(teacher_ckpt), model_cfg, device)
prompts = load_prompt_bank(out_root / "manifests/evaluation_prompts_pilot.json")


In [ ]:
grid = list(itertools.product([3, 4, 5], [4, 8], [2, 4], [True, False]))
rows = []
for horizon, topk, beam, deterministic in grid:
    cfg_sel = SelectedDecodingConfig(
        horizon=horizon,
        first_step_top_k=topk,
        beam_width=beam,
        deterministic=deterministic,
        repetition_ngram=4,
        repetition_penalty=0.8,
    )
    out = selected_next_token(teacher, prompts[0], cfg_sel, device)
    rows.append(
        {
            "horizon": horizon,
            "top_k": topk,
            "beam_width": beam,
            "deterministic": deterministic,
            "selected_token": out["chosen_next_token"],
            "best_branch_score": out["best_branch_score"],
        }
    )

ab_df = pd.DataFrame(rows).sort_values(["horizon", "top_k", "beam_width", "deterministic"])
display(ab_df)

out_csv = ROOT / "GitHub/data/outputs/transformer_conan_doyle/evaluation/csv/ablation_pilot.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
ab_df.to_csv(out_csv, index=False)
out_csv
